# **EXPERIMENTO - 02**

REGRESSÃO 02 - DecisionTreeRegressor

In [137]:
# Import das bibliotecas e APIs.

import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import RFECV

In [138]:
# Import do dataset "winequality-red.csv".

df = np.loadtxt("winequality-red.csv", delimiter=",", skiprows=1)

In [139]:
print (df.shape)

(1599, 12)


In [140]:
nomes_features = np.array(["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides", "free sulfur dioxide",
"total sulfur dioxide", "density","pH", "sulphates", "alcohol"])

In [141]:
# Criação da matriz X com as variáveis de entrada e o vetor y com a variável de saída.

X = df[:,0:11]
y = df[:,11]

In [142]:
# Separação do conjunto de treino e teste (70%/30%).

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.3, random_state=42)

In [143]:
estimator_rfe = DecisionTreeRegressor(random_state=42)

rfe_seletor = RFECV(estimator=estimator_rfe, step=1, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

In [144]:
rfe_seletor.fit(X_treino, y_treino)

,estimator,DecisionTreeR...ndom_state=42)
,step,1
,min_features_to_select,1
,cv,5
,scoring,'neg_mean_squared_error'
,verbose,0
,n_jobs,-1
,importance_getter,'auto'
,criterion,'squared_error'
,splitter,'best'
,max_depth,None


In [145]:
X_treino_selecionado = rfe_seletor.transform(X_treino)
X_teste_selecionado = rfe_seletor.transform(X_teste)

In [146]:
feat_selecionada = rfe_seletor.get_support()
nomes_selecionados = nomes_features[feat_selecionada]

In [147]:
print("\n### Seleção de Features com RFECV (Decission Tree) ###")
print(f"Total de features iniciais: {len(nomes_features)}")
print(f"Features mantidas pelo RFECV: {len(nomes_selecionados)}")
print(f"Ranking das features (1=melhor): {rfe_seletor.ranking_}") # 
print("\nTop 5 Features Selecionadas:")
print(nomes_selecionados[:5])


### Seleção de Features com RFECV (Decission Tree) ###
Total de features iniciais: 11
Features mantidas pelo RFECV: 1
Ranking das features (1=melhor): [11  3 10  6  4  8  5  9  7  2  1]

Top 5 Features Selecionadas:
['alcohol']


In [148]:
modelo = DecisionTreeRegressor()
parametros = {"max_depth": [2, 5, 10, 20, 50]} 

In [149]:
scoring = "neg_mean_squared_error"

grid_search = GridSearchCV(estimator=modelo, param_grid=parametros, scoring=scoring, cv=5, verbose=1,n_jobs=-1)

In [150]:
print("\n### Iniciando GridSearchCV para encontrar o melhor hiperparâmtero (max_depth) ###")
grid_search.fit(X_treino_selecionado, y_treino)

melhor_hiperparametro = grid_search.best_params_['max_depth']
melhor_neg_mse = grid_search.best_score_
melhor_mse = -melhor_neg_mse 


### Iniciando GridSearchCV para encontrar o melhor hiperparâmtero (max_depth) ###
Fitting 5 folds for each of 5 candidates, totalling 25 fits


In [151]:
print("\n### Resultados da Otimização ###")
print(f"melhor hiperparâmtero (max_depth): {melhor_hiperparametro}")
print(f"Melhor MSE (médio na validação cruzada): {melhor_mse:.4f}")


### Resultados da Otimização ###
melhor hiperparâmtero (max_depth): 2
Melhor MSE (médio na validação cruzada): 0.4918


In [152]:
modelo_final_DecisionTreeRegressor = grid_search.best_estimator_
y_pred_final = modelo_final_DecisionTreeRegressor.predict(X_teste_selecionado)

rmse_final = np.sqrt(mean_squared_error(y_teste, y_pred_final))
r2_final = grid_search.best_estimator_.score(X_teste_selecionado, y_teste)

print("\n### Métricas no Conjunto de Teste ###")
print(f"Modelo: DecisionTreeRegressor (max_depth={melhor_hiperparametro})")


### Métricas no Conjunto de Teste ###
Modelo: DecisionTreeRegressor (max_depth=2)


In [154]:
modelo_final_DecisionTreeRegressor = grid_search.best_estimator_
melhor_hiperparametro = grid_search.best_params_["max_depth"]

y_pred_teste = modelo_final_DecisionTreeRegressor.predict(X_teste_selecionado)

mse_teste = mean_squared_error(y_teste, y_pred_teste)

r2_teste = r2_score(y_teste, y_pred_teste)

print("\n#############################################")
print("### TREINAMENTO E AVALIAÇÃO FINAL (DecisionTreeRegressor) ###")
print("#############################################")
print(f"Modelo Final: DecisionTreeRegressor")
print(f"Hiperparâmetro Otimizado: {melhor_hiperparametro}")
print(f"Número de Features Utilizadas: {X_treino_selecionado.shape[1]}")
print("\n--- Métricas de Desempenho no Conjunto de Teste ---")

print(f"A. Métrica MSE (Erro Quadrático Médio): {mse_teste:.4f}")
print(f"B. Métrica R² (Coeficiente de Determinação): {r2_teste:.4f}")


#############################################
### TREINAMENTO E AVALIAÇÃO FINAL (DecisionTreeRegressor) ###
#############################################
Modelo Final: DecisionTreeRegressor
Hiperparâmetro Otimizado: 2
Número de Features Utilizadas: 1

--- Métricas de Desempenho no Conjunto de Teste ---
A. Métrica MSE (Erro Quadrático Médio): 0.5167
B. Métrica R² (Coeficiente de Determinação): 0.1850
